In [3]:
import nest_asyncio
nest_asyncio.apply()

In [4]:
import os
import asyncio
import logging

# from raganything import RAGAnything
from lightrag import LightRAG
from lightrag.llm.openai import openai_complete_if_cache, openai_embed
from lightrag.utils import EmbeddingFunc, setup_logger,logger ,wrap_embedding_func_with_attrs, set_verbose_debug

In [7]:
from dotenv import load_dotenv

# env
load_dotenv("../.env", override=True)

# Logger

setup_logger("lightrag", level="INFO")

# Config
WORKING_DIR = "./rag_storage"
if not os.path.exists(WORKING_DIR):
    os.mkdir(WORKING_DIR)

## Setup Lightrag

In [8]:
# LLM Model Function
async def llm_model_func(
    prompt, system_prompt=None, history_messages=[], keyword_extraction=False, **kwargs
) -> str:
    return await openai_complete_if_cache(
        os.getenv("LLM_MODEL"),
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=os.getenv("LLM_BINDING_API_KEY"),
        base_url=os.getenv("LLM_BINDING_HOST"),
        **kwargs,
    )

async def print_stream(stream):
    async for chunk in stream:
        if chunk:
            print(chunk, end="", flush=True)


# Lightrag
async def initialize_rag():

    embedding_dim = int(os.getenv("EMBEDDING_DIM", 1536))
    token_limit = int(os.getenv("EMBEDDING_TOKEN_LIMIT", 8192))
    model_name = os.getenv("EMBEDDING_MODEL")

    # Step 1: define raw embedding function
    async def raw_embedding_func(texts):
        return await openai_embed.func(
            texts,
            api_key=os.getenv("LLM_BINDING_API_KEY"),
            base_url=os.getenv("LLM_BINDING_HOST"),
            model=model_name,
        )

    # Step 2: wrap embedding function
    embedding_func = wrap_embedding_func_with_attrs(
        embedding_dim=embedding_dim,
        max_token_size=token_limit,
        model_name=model_name,
    )(raw_embedding_func)

    # Step 3: initialize LightRAG
    rag = LightRAG(
        working_dir=WORKING_DIR,
        llm_model_func=llm_model_func,
        embedding_func=embedding_func,
        kv_storage="PGKVStorage",
        vector_storage="PGVectorStorage",
        graph_storage="Neo4JStorage",
    )

    await rag.initialize_storages()
    return rag


In [9]:
# Rag initialization
rag = await initialize_rag() 

INFO: PostgreSQL table: LIGHTRAG_VDB_ENTITY_text_embedding_3_small_1536d
INFO: PostgreSQL table: LIGHTRAG_VDB_RELATION_text_embedding_3_small_1536d
INFO: PostgreSQL table: LIGHTRAG_VDB_CHUNKS_text_embedding_3_small_1536d
INFO: HNSW vector index idx_c79820f98cd4_hnsw_cosine already exists on table LIGHTRAG_VDB_ENTITY_text_embedding_3_small_1536d
INFO: HNSW vector index idx_a6b494f856e1_hnsw_cosine already exists on table LIGHTRAG_VDB_RELATION_text_embedding_3_small_1536d
INFO: HNSW vector index idx_036611e7ab3b_hnsw_cosine already exists on table LIGHTRAG_VDB_CHUNKS_text_embedding_3_small_1536d
INFO: [base] Connected to neo4j at neo4j://127.0.0.1:7687
INFO: [base] Ensured B-Tree index on entity_id for base in neo4j
INFO: [base] Found existing index 'entity_id_fulltext_idx_base' with state: ONLINE
INFO: [base] Full-text index 'entity_id_fulltext_idx_base' already exists and is online. Skipping recreation.


In [10]:
# Test embedding function
test_text = ["This is a test string for embedding."]
embedding = await rag.embedding_func(test_text)
embedding_dim = embedding.shape[1]
print("\n=======================")
print("Test embedding function")
print("========================")
print(f"Test dict: {test_text}")
print(f"Detected embedding dimension: {embedding_dim}\n\n")

INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)



Test embedding function
Test dict: ['This is a test string for embedding.']
Detected embedding dimension: 1536




## Function helper

In [13]:
# Function helper
# Used
import zipfile
from xml.etree import ElementTree as ET
from pathlib import Path
import pdfplumber

NS = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
VMERGE = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val'

def _read_docx(filepath):
    with zipfile.ZipFile(filepath) as z:
        root = ET.fromstring(z.read('word/document.xml'))
    lines = []
    for child in root.find('.//w:body', NS):
        tag = child.tag.split('}')[-1]
        if tag == 'p':
            text = ''.join(
                r.find('w:t', NS).text for r in child.findall('.//w:r', NS)
                if r.find('w:t', NS) is not None and r.find('w:t', NS).text
            ).strip()
            if text:
                lines.append(text)
        elif tag == 'tbl':
            for row in child.findall('w:tr', NS):
                cells = []
                for cell in row.findall('w:tc', NS):
                    vm = cell.find('.//w:vMerge', NS)
                    if vm is not None and vm.get(VMERGE) != 'restart':
                        continue
                    text = ''.join(
                        r.find('w:t', NS).text
                        for p in cell.findall('.//w:p', NS)
                        for r in p.findall('.//w:r', NS)
                        if r.find('w:t', NS) is not None and r.find('w:t', NS).text
                    ).strip()
                    if text:
                        cells.append(text)
                if cells:
                    lines.append(' | '.join(cells))
    return '\n'.join(lines)

def _read_pdf(filepath):
    with pdfplumber.open(filepath) as pdf:
        return '\n'.join(
            page.extract_text() for page in pdf.pages
            if page.extract_text()
        )

def read_folder(folder_path):
    results = []
    for path in Path(folder_path).rglob('*'):
        suffix = path.suffix.lower()
        if suffix == '.docx':
            text = _read_docx(path)
        elif suffix == '.pdf':
            text = _read_pdf(path)
        else:
            continue
        results.append({'filename': path.name, 'text': text})
        print(f"✓ {path.name} ({len(text)} karakter)")
    return results


def extract_text_from_file(uploaded_file) -> str:
    """Save uploaded file to temp then read."""
    suffix = Path(uploaded_file.name).suffix.lower()
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(uploaded_file.read())
        tmp_path = tmp.name
    try:
        if suffix == ".docx":
            return _read_docx(tmp_path)
        elif suffix == ".pdf":
            return _read_pdf(tmp_path)
        elif suffix == ".txt":
            with open(tmp_path, "r", encoding="utf-8") as f:
                return f.read()
        else:
            return ""
    finally:
        os.unlink(tmp_path)


# Load SKJ Json
def load_all_skj():
    import json
    skj_folder = "../data/skj_documents_json"

    skj_dict = {}
    for json_file in skj_folder.glob("*.json"):
        try:
            with open(json_file, "r", encoding="utf-8") as f:
                data = json.load(f)

                if "profil_jabatan" in data:
                    jabatan = data["profil_jabatan"].get(
                        "nama_jabatan", json_file.stem
                    )
                elif "nama_jabatan" in data:
                    jabatan = data["nama_jabatan"]
                else:
                    jabatan = json_file.stem

                skj_dict[jabatan] = data

        except Exception as e:
            print(f"Gagal load {json_file.name}: {e}")

    return skj_dict


SCORE_LABELS = {
    "n1_kesesuaian_judul":   "Kesesuaian Judul dengan Tema",
    "n2_kesesuaian_isi":     "Kesesuaian Isi dengan Judul & Tema",
    "n3_sistematika":        "Sistematika Penulisan",
    "n4_ketajaman_analisis": "Ketajaman Analisis",
    "n5_penggunaan_bahasa":  "Penggunaan Bahasa",
}
SCORE_WEIGHTS = {
    "n1_kesesuaian_judul":   1,
    "n2_kesesuaian_isi":     1,
    "n3_sistematika":        1,
    "n4_ketajaman_analisis": 2,   # bobot 2x
    "n5_penggunaan_bahasa":  1,
}

### Prompts


In [14]:
# ── Prompt ────────────────────────────────────────────────────────────────────
QUERY_KEYWORDS = """
Penilaian Penulisan Makalah
Form. 1 Penilaian Penulisan Makalah
{selected_jabatan}
Kesesuaian Judul dengan Tema
Kesesuaian Isi Makalah dengan Judul dan Tema
Sistematika Penulisan
Ketajaman Analisis
Penggunaan Bahasa dalam Penulisan Makalah
Bobot Penilaian Penulisan Makalah
Format Penulisan Makalah
Struktur Makalah (Pendahuluan, Analisis dan Sintesis, Rencana Strategis, Plan Of Action, Konklusi)
Penilaian Kompetensi Teknis/Bidang
Kompetensi Bidang
Panitia Seleksi
"""

PROMPT_KONTEKS = """
Anda adalah asisten yang bertugas mengumpulkan konteks relevan untuk penilaian makalah.

Berdasarkan jabatan '{selected_jabatan}', berikan ringkasan singkat tentang:
1. Deskripsi jabatan dan kompetensi yang diperlukan
2. Kriteria penilaian utama untuk posisi ini
3. Standar kualitas yang diharapkan dalam penulisan makalah

Berikan jawaban dalam format paragraf singkat, fokus pada poin-poin penting yang akan membantu dalam evaluasi makalah.
"""

PROMPT_PENILAIAN = """
---Role---

Anda adalah evaluator akademik sebagai Panitia Seleksi yang bertugas menilai kualitas substansi makalah secara objektif dan sistematis. Penilaian harus didasarkan hanya pada isi makalah yang tersedia, dengan mempertimbangkan konteks jabatan yang dituju.

---Goal---

Melakukan penilaian terhadap makalah berdasarkan kriteria penilaian yang telah ditentukan, memberikan skor numerik untuk setiap kriteria, serta menyusun justifikasi yang jelas dan berbasis bukti dari isi makalah.

---Konteks Jabatan---

{assessment_context}

---Instructions---

1. Baca dan pahami isi makalah secara menyeluruh.
2. Tinjau konteks jabatan di atas sebagai acuan penilaian.
3. Lakukan penilaian terhadap setiap kriteria dengan memberikan skor antara 40 sampai 100.
4. Setiap skor harus disertai justifikasi yang menjelaskan alasan pemberian skor.
5. Penilaian harus objektif, sistematis, dan berbasis isi makalah.
6. Gunakan bahasa formal dan akademik.
7. Jangan menggunakan informasi di luar isi makalah.
8. Jika informasi dalam makalah terbatas, tetap berikan skor dengan menjelaskan keterbatasan informasi tersebut.
9. Hitung nilai akhir menggunakan rumus yang telah ditentukan.
10. Output harus dalam format JSON yang valid dan tidak boleh mengandung teks tambahan di luar JSON.

---Assessment Criteria---

A. Penulisan Makalah

1. Kesesuaian judul dengan tema
Menilai kesesuaian antara judul dan tema yang dibahas dalam makalah.

2. Kesesuaian isi makalah dengan judul dan tema
Menilai kesesuaian antara isi makalah dengan judul dan tema.

3. Sistematika penulisan
Menilai keteraturan struktur penulisan dan alur pembahasan.

4. Ketajaman analisis
Menilai kedalaman pemikiran, argumentasi, dan kemampuan analisis terhadap permasalahan.

5. Penggunaan bahasa dalam penulisan makalah
Menilai kejelasan, ketepatan, dan konsistensi penggunaan bahasa.

---Scoring Rules---

- Skor minimum: 40
- Skor maksimum: 100
- Semua skor harus berupa bilangan bulat
- Ketajaman analisis memiliki bobot dua kali lipat dalam nilai akhir

---Makalah---

{makalah_text}

---Output Format---

Hasil harus dalam format JSON berikut:
{{
  "Ringkasan": "ringkasan isi makalah yang menjelaskan tentang keseluruhan makalah",

  "scores": {{
    "n1_kesesuaian_judul": 0,
    "n2_kesesuaian_isi": 0,
    "n3_sistematika": 0,
    "n4_ketajaman_analisis": 0,
    "n5_penggunaan_bahasa": 0
  }},

  "justification": {{
    "n1_kesesuaian_judul": "",
    "n2_kesesuaian_isi": "",
    "n3_sistematika": "",
    "n4_ketajaman_analisis": "",
    "n5_penggunaan_bahasa": ""
  }},

  "evidence": {{
    "n1_kesesuaian_judul": "",
    "n2_kesesuaian_isi": "",
    "n3_sistematika": "",
    "n4_ketajaman_analisis": "",
    "n5_penggunaan_bahasa": ""
  }},

  "final_score": 0
}}
"""

In [15]:
# ── Helpers ───────────────────────────────────────────────────────────────────
import json


def compute_final_score(scores: dict) -> float:
    total_weight = sum(SCORE_WEIGHTS.values())
    weighted_sum = sum(scores.get(k, 0) * w for k, w in SCORE_WEIGHTS.items())
    return round(weighted_sum / total_weight, 1)


def parse_response(raw: str) -> dict:
    """Try to extract JSON from LLM response."""
    raw = raw.strip()
    # strip markdown fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())

In [16]:
# ── Two-stage RAG functions ───────────────────────────────────────────────────
from lightrag import QueryParam

#1 Retrieve konteks dari knowledge base
async def retrieve_assessment_context(rag, selected_jabatan: str, query_mode: str) -> str:
    """
    Tahap 1: Retrieve konteks jabatan, kriteria penilaian, renstra, visi misi dari knowledge base.
    """
    try:
        context_response = await rag.aquery(
            query=QUERY_KEYWORDS.format(selected_jabatan=selected_jabatan),
            param=QueryParam(
                mode=query_mode,
                user_prompt=PROMPT_KONTEKS.format(selected_jabatan=selected_jabatan),
                # enable_rerank=False,
                # max_token_for_context=2000,
            ),
        )
        return context_response if context_response else "Konteks jabatan tidak ditemukan dalam knowledge base."
    except Exception as e:
        return f"Error retrieving context: {str(e)}"


async def evaluate_paper_with_context(
    rag, 
    makalah_text: str, 
    assessment_context: str,
    query_mode: str
) -> dict:
    """
    Tahap 2: Evaluate makalah berdasarkan konteks yang telah diambil.
    """
    evaluation_prompt = PROMPT_PENILAIAN.format(
        assessment_context=assessment_context,
        makalah_text=makalah_text
    )
    
    eval_response = await rag.aquery(
        query="Penilaian Makalah: Kesesuaian Isi, Sistematika, Ketajaman Analisis, Penggunaan Bahasa",
        param=QueryParam(
            mode=query_mode,
            user_prompt=evaluation_prompt,
            # enable_rerank=False,
            # max_token_for_context=3000,
        ),
    )
    return parse_response(eval_response)


### Baca data makalah dari Minio